# Body Measurement Inference — EfficientNet-B2




In [ ]:
import subprocess, sys

pkgs = [
    ["numpy==1.26.4", "--force-reinstall"],
    ["Pillow==10.4.0", "--force-reinstall"],  
    ["onnxruntime"],
    ["onnx"],
    ["onnxscript"],
    ["mediapipe==0.9.3"],
    ["rembg[gpu]"],
]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkg,
                   capture_output=True)

print(" Dependencies installed")


In [7]:
import os, pickle, warnings, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import cv2
import torch
import torch.nn as nn
import onnxruntime as ort
from torchvision import transforms, models
warnings.filterwarnings("ignore")

#rembg (primary segmenter)
try:
    from rembg import remove as rembg_remove, new_session
    _rembg_session = new_session("u2net_human_seg")  # human-specific model
    _REMBG_AVAILABLE = True
except ImportError:
    _REMBG_AVAILABLE = False

if not _REMBG_AVAILABLE:
    raise RuntimeError("rembg not installed. Run Cell 1 first, then restart kernel.")

print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device    : {DEVICE}")
print(f"rembg     : OK  (model: u2net_human_seg)")

PyTorch   : 2.10.0+cu128
CUDA      : True
Device    : cuda
rembg     : OK  (model: u2net_human_seg)


In [8]:
MODEL_PATH  = '/kaggle/input/datasets/aminalawal/model-infor/bodym_final_model.pth'
SCALER_PATH = '/kaggle/input/datasets/aminalawal/model-infor/label_scaler.pkl'
OUTPUT_DIR  = '/kaggle/working'
ONNX_PATH   = os.path.join(OUTPUT_DIR, 'bodym_model.onnx')
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
IMG_SIZE      = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
 
MEASUREMENT_COLS = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm',
    'height', 'hip', 'leg-length', 'shoulder-breadth',
    'shoulder-to-crotch', 'thigh', 'waist', 'wrist'
]
NUM_OUTPUTS = len(MEASUREMENT_COLS)
 
print(f"Config ready — {NUM_OUTPUTS} measurements")

Config ready — 14 measurements


In [9]:
class DualBranchBodyM(nn.Module):
    """Dual-branch EfficientNet-B2 for body measurement regression."""

    def __init__(self, num_outputs: int, dropout: float = 0.3):
        super().__init__()
        self.front_branch = self._build_backbone()
        self.side_branch  = self._build_backbone()
        feat_dim  = 1408          # EfficientNet-B2 feature size
        fused_dim = feat_dim * 2 + 3   # 2 × image features + aux (gender, H, W)
        self.head = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_outputs),
        )

    @staticmethod
    def _build_backbone() -> nn.Module:
        backbone = models.efficientnet_b2(weights=None)
        backbone.classifier = nn.Identity()
        return backbone

    def forward(self, front_img: torch.Tensor,
                side_img: torch.Tensor, aux: torch.Tensor) -> torch.Tensor:
        front_feat = self.front_branch(front_img)
        side_feat  = self.side_branch(side_img)
        fused      = torch.cat([front_feat, side_feat, aux], dim=1)
        return self.head(fused)


class DualBranchONNXWrapper(nn.Module):
    """Flattens the three inputs into a single tensor for ONNX export."""

    def __init__(self, model: DualBranchBodyM):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        front = x[:, 0:3, :, :]
        side  = x[:, 3:6, :, :]
        aux   = x[:, 6:9, 0, 0]
        return self.model(front, side, aux)


print(" Architecture defined")


 Architecture defined


In [10]:
model = DualBranchBodyM(num_outputs=NUM_OUTPUTS).to(DEVICE)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Model loaded — Test MAE: {checkpoint['test_mean_mae_cm']:.3f} cm")
model = model.to(DEVICE) 
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)
print(" Scaler loaded")


Model loaded — Test MAE: 1.191 cm
 Scaler loaded


In [11]:
#  SILHOUETTE EXTRACTION PIPELINE
#  Two stages: segment → clean → resize + normalize (matches val/test transform)


def _segment_rembg(img_rgb: np.ndarray) -> np.ndarray:
    """
    Stage 1 — rembg with u2net_human_seg (human-specific model).
    Returns a uint8 binary mask (0 / 255).
    """
    img_pil = Image.fromarray(img_rgb)
    out_pil = rembg_remove(img_pil, session=_rembg_session)
    alpha   = np.array(out_pil)[:, :, 3]
    mask    = (alpha > 127).astype(np.uint8) * 255
    return mask


def _clean_mask(mask: np.ndarray) -> np.ndarray:
    """
    Minimal cleanup — only removes small noise blobs.
    rembg u2net_human_seg already produces clean masks so heavy
    morphological operations do more harm than good.
    """
    # Remove small isolated noise pixels (e.g. the dot visible in the raw mask)
    # by keeping only connected components larger than 500 pixels
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    clean = np.zeros_like(mask)
    for i in range(1, num_labels):  # skip 0 (background)
        if stats[i, cv2.CC_STAT_AREA] > 500:
            clean[labels == i] = 255
    return clean


def prepare_silhouette(image_path: str, return_debug: bool = False):
    """Full pipeline with error handling."""
    # --- load & convert to RGB ---
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Stage 1 — segmentation
    raw_mask = _segment_rembg(img_rgb)
    debug_mask = raw_mask.copy()

    # Stage 2 — morphological cleanup (use the gentler version above)
    clean_mask = _clean_mask(raw_mask)

    # Stage 3 — apply transform
    sil_rgb = cv2.cvtColor(clean_mask, cv2.COLOR_GRAY2RGB)
    sil_pil = Image.fromarray(sil_rgb)

    tfm = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    sil_tensor = tfm(sil_pil).unsqueeze(0)  # (1, 3, 224, 224)

    if return_debug:
        return sil_tensor, sil_pil, debug_mask
    return sil_tensor, sil_pil


def show_silhouette_pipeline(front_path: str, side_path: str):
    """
    Diagnostic plot: Original | Raw Mask | Cleaned → Final (224×224)
    """
    fig, axes = plt.subplots(2, 3, figsize=(12, 10))
    fig.suptitle("Silhouette Extraction Pipeline", fontsize=15, fontweight="bold")

    col_titles = ["Original", "Raw Mask", "Cleaned → Final (224×224)"]
    for ax, t in zip(axes[0], col_titles):
        ax.set_title(t, fontsize=11)

    row_labels = ["Frontal", "Lateral"]
    paths      = [front_path, side_path]

    for row, (label, path) in enumerate(zip(row_labels, paths)):
        axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, labelpad=10)

        img_bgr = cv2.imread(path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        axes[row, 0].imshow(img_rgb)

        raw_mask = _segment_rembg(img_rgb)
        axes[row, 1].imshow(raw_mask, cmap="gray")

        clean_mask  = _clean_mask(raw_mask)
        sil_resized = cv2.resize(clean_mask, (IMG_SIZE, IMG_SIZE),
                                 interpolation=cv2.INTER_AREA)
        axes[row, 2].imshow(sil_resized, cmap="gray")

    for ax in axes.flat:
        ax.axis("off")

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "silhouette_pipeline.png")
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Pipeline plot saved → {out}")


print(" Silhouette pipeline defined")

 Silhouette pipeline defined


In [12]:
# Image transform (must match training)
_img_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Uncertainty thresholds (from training-set std across predictions)
HIGH_UNCERTAINTY = {'chest', 'hip', 'waist', 'thigh', 'bicep'}


def _build_aux_tensor(gender: str, height_cm: float, weight_kg: float) -> torch.Tensor:
    """Encode auxiliary inputs the same way the training script did."""
    gender_val = 1.0 if gender.lower().strip() in ('female', 'f', 'woman') else 0.0
    aux = torch.tensor([[gender_val, height_cm / 200.0, weight_kg / 150.0]],
                       dtype=torch.float32)
    return aux


def _decode_predictions(raw: np.ndarray) -> dict:
    """Inverse-transform scaler output → measurement dict (cm)."""
    raw2d = raw.reshape(1, -1)
    cm    = scaler.inverse_transform(raw2d)[0]
    return {name: float(round(val, 1)) for name, val in zip(MEASUREMENT_COLS, cm)}


# ─────────────────────────────────────────────────────────────────────────────
#  PyTorch inference
# ─────────────────────────────────────────────────────────────────────────────
def predict_pytorch(front_path: str, side_path: str,
                    gender: str, height_cm: float, weight_kg: float,
                    segmenter: str = "mediapipe",
                    show_silhouettes: bool = True) -> dict:
  
    print("Step 1/3 — Extracting silhouettes …")
    front_tensor, front_sil = prepare_silhouette(front_path)
    side_tensor,  side_sil  = prepare_silhouette(side_path)

    if show_silhouettes:
        fig, axes = plt.subplots(1, 2, figsize=(8, 5))
        axes[0].imshow(front_sil); axes[0].set_title("Silhouette — Frontal"); axes[0].axis("off")
        axes[1].imshow(side_sil);  axes[1].set_title("Silhouette — Lateral");  axes[1].axis("off")
        plt.tight_layout()
        out = os.path.join(OUTPUT_DIR, "silhouettes.png")
        plt.savefig(out, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"   Silhouettes saved → {out}")

    print("Step 2/3 — Preprocessing …")
    front_t = front_tensor.to(DEVICE)
    side_t  = side_tensor.to(DEVICE)
    aux_t   = _build_aux_tensor(gender, height_cm, weight_kg).to(DEVICE)
    print("   Tensors ready")

    print("Step 3/3 — Running PyTorch model …")
    with torch.no_grad():
        # Move everything to the same device
        front_t = front_tensor.to(DEVICE)
        side_t = side_tensor.to(DEVICE)
        aux_t = _build_aux_tensor(gender, height_cm, weight_kg).to(DEVICE)
        raw = model(front_t, side_t, aux_t).cpu().numpy()
    print("   Predictions complete")

    return _decode_predictions(raw)


# ─────────────────────────────────────────────────────────────────────────────
#  ONNX inference
# ─────────────────────────────────────────────────────────────────────────────
def predict_onnx(front_path: str, side_path: str,
                 gender: str, height_cm: float, weight_kg: float,
                 segmenter: str = "mediapipe",
                 show_silhouettes: bool = False) -> dict:
    """
    Run inference via the exported ONNX runtime (faster on CPU / mobile).
    ONNX must be exported first (see Cell 8).
    """
    if not os.path.exists(ONNX_PATH):
        raise FileNotFoundError("ONNX model not found. Run Cell 8 first.")

    front_tensor, _ = prepare_silhouette(front_path, segmenter=segmenter)
    side_tensor,  _ = prepare_silhouette(side_path,  segmenter=segmenter)
    aux             = _build_aux_tensor(gender, height_cm, weight_kg)

    # Pack into the single 9-channel tensor the ONNX wrapper expects
    # Channels 0-2: front, 3-5: side, 6-8: aux (broadcast to spatial dims)
    aux_spatial = aux.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, IMG_SIZE, IMG_SIZE)
    packed = torch.cat([front_tensor, side_tensor, aux_spatial], dim=1).numpy()

    sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
    raw  = sess.run(['measurements'], {'input': packed})[0]
    return _decode_predictions(raw)


In [13]:
onnx_wrapper = DualBranchONNXWrapper(model).cpu().eval()
dummy_input  = torch.zeros(1, 9, IMG_SIZE, IMG_SIZE)
 
torch.onnx.export(
    onnx_wrapper,
    dummy_input,
    ONNX_PATH,
    input_names   = ['input'],
    output_names  = ['measurements'],
    dynamic_axes  = {'input': {0: 'batch_size'}, 'measurements': {0: 'batch_size'}},
    opset_version = 18,
    verbose       = False,
)
 

model.to(DEVICE).eval()
 
print(f"ONNX exported → {ONNX_PATH}")
print(f"  File size : {os.path.getsize(ONNX_PATH)/1024/1024:.1f} MB")

W0502 12:28:44.966000 180 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0502 12:28:44.967000 180 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0502 12:28:44.969000 180 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0502 12:28:44.971000 180 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


ONNX exported → /kaggle/working/bodym_model.onnx
  File size : 2.1 MB


In [14]:
_img_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

HIGH_UNCERTAINTY = {'chest', 'hip', 'waist', 'thigh', 'bicep'}


def _build_aux_tensor(gender: str, height_cm: float, weight_kg: float) -> torch.Tensor:
    # Must exactly match BodyMDataset.GENDER_MAP: female=0, male=1
    gender_map = {'female': 0, 'male': 1, 'f': 0, 'm': 1}
    gender_val = float(gender_map.get(gender.lower().strip(), 0))

    # Raw values — no normalization — exactly as in training
    return torch.tensor([[gender_val, height_cm, weight_kg]], dtype=torch.float32)


def _decode_predictions(raw: np.ndarray) -> dict:
    raw2d = raw.reshape(1, -1)
    cm    = scaler.inverse_transform(raw2d)[0]
    return {name: float(round(val, 1)) for name, val in zip(MEASUREMENT_COLS, cm)}


def predict_pytorch(front_path: str, side_path: str,
                    gender: str, height_cm: float, weight_kg: float,
                    show_silhouettes: bool = True) -> dict:
    print("Step 1/3 — Extracting silhouettes …")
    front_tensor, front_sil = prepare_silhouette(front_path)
    side_tensor,  side_sil  = prepare_silhouette(side_path)

    if show_silhouettes:
        fig, axes = plt.subplots(1, 2, figsize=(8, 5))
        axes[0].imshow(front_sil); axes[0].set_title("Silhouette — Frontal"); axes[0].axis("off")
        axes[1].imshow(side_sil);  axes[1].set_title("Silhouette — Lateral");  axes[1].axis("off")
        plt.tight_layout()
        out = os.path.join(OUTPUT_DIR, "silhouettes.png")
        plt.savefig(out, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"   Silhouettes saved → {out}")

    print("Step 2/3 — Preprocessing …")
    front_t = front_tensor.to(DEVICE)
    side_t  = side_tensor.to(DEVICE)
    aux_t   = _build_aux_tensor(gender, height_cm, weight_kg).to(DEVICE)
    print("   Tensors ready")

    print("Step 3/3 — Running PyTorch model …")
    model.to(DEVICE).eval()
    with torch.no_grad():
        raw = model(front_t, side_t, aux_t).cpu().numpy()
    print("   Predictions complete")

    return _decode_predictions(raw)


def predict_onnx(front_path: str, side_path: str,
                 gender: str, height_cm: float, weight_kg: float,
                 show_silhouettes: bool = False) -> dict:
    if not os.path.exists(ONNX_PATH):
        raise FileNotFoundError("ONNX model not found. Run the ONNX export cell first.")

    front_tensor, _ = prepare_silhouette(front_path)
    side_tensor,  _ = prepare_silhouette(side_path)
    aux             = _build_aux_tensor(gender, height_cm, weight_kg)

    aux_spatial = aux.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, IMG_SIZE, IMG_SIZE)
    packed = torch.cat([front_tensor, side_tensor, aux_spatial], dim=1).numpy()

    sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
    raw  = sess.run(['measurements'], {'input': packed})[0]
    return _decode_predictions(raw)


def print_results_table(predictions: dict, gender: str,
                        height_cm: float, weight_kg: float) -> None:
    label = gender.capitalize()
    print()
    print("=" * 50)
    print("  PREDICTED BODY MEASUREMENTS")
    print(f"  Gender: {label}  |  Height: {height_cm}cm  |  Weight: {weight_kg}kg")
    print("=" * 50)
    print(f"  {'Measurement':<22} {'Predicted (cm)':>14}")
    print("  " + "-" * 38)
    for name, val in predictions.items():
        flag = " " if name in HIGH_UNCERTAINTY else ""
        print(f"  {name:<22} {val:>14.1f}{flag}")
   
    print()


def display_results(predictions: dict, gender: str,
                    height_cm: float, weight_kg: float,
                    front_path: str) -> None:
    names  = list(predictions.keys())
    values = list(predictions.values())
    colors = ['#e74c3c' if n in HIGH_UNCERTAINTY else '#2980b9' for n in names]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                             gridspec_kw={'width_ratios': [1, 2]})

    img = Image.open(front_path).convert('RGB')
    axes[0].imshow(img)
    axes[0].set_title("Input — Frontal", fontsize=12)
    axes[0].axis('off')

    bars = axes[1].barh(names, values, color=colors, edgecolor='white', linewidth=0.5)
    axes[1].set_xlabel("cm", fontsize=11)
    axes[1].set_title(
        f"Predicted Measurements — {gender.capitalize()} | "
        f"{height_cm} cm | {weight_kg} kg",
        fontsize=12, fontweight='bold'
    )
    axes[1].invert_yaxis()
    for bar, val in zip(bars, values):
        axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                     f"{val:.1f}", va='center', fontsize=9)

    patch_high = mpatches.Patch(color='#e74c3c', label='Higher uncertainty (⚠)')
    patch_norm = mpatches.Patch(color='#2980b9', label='Standard confidence')
    axes[1].legend(handles=[patch_norm, patch_high], loc='lower right', fontsize=9)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "predictions.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f" Chart saved → {out}")


def save_predictions_json(predictions: dict, gender: str,
                          height_cm: float, weight_kg: float) -> str:
    payload = {
        "gender": gender, "height_cm": height_cm, "weight_kg": weight_kg,
        "measurements_cm": predictions
    }
    out = os.path.join(OUTPUT_DIR, "predictions.json")
    with open(out, 'w') as f:
        json.dump(payload, f, indent=2)
    print(f" Predictions saved → {out}")
    return out


print(" Inference helpers defined")

 Inference helpers defined


In [15]:
# ════════════════════════════════════════════════════
#  ▶  FILL IN YOUR DETAILS HERE
# ════════════════════════════════════════════════════
FRONT_PHOTO = ''
SIDE_PHOTO  = ''
 
GENDER    = 'female'
HEIGHT_CM = 163.0
WEIGHT_KG = 65.0
 
print(f"Inputs set  →  {GENDER}, {HEIGHT_CM} cm, {WEIGHT_KG} kg")
print(f"  Front : {FRONT_PHOTO}")
print(f"  Side  : {SIDE_PHOTO}")

Inputs set  →  female, 163.0 cm, 65.0 kg
  Front : 
  Side  : 


In [ ]:
show_silhouette_pipeline(FRONT_PHOTO, SIDE_PHOTO)


In [ ]:
print("=== PyTorch Inference ===")
pytorch_predictions = predict_pytorch(
    FRONT_PHOTO, SIDE_PHOTO, GENDER, HEIGHT_CM, WEIGHT_KG,
    show_silhouettes=True,
)
print_results_table(pytorch_predictions, GENDER, HEIGHT_CM, WEIGHT_KG)